# Rendu cosmos sur GPU (Kaggle / Colab)

**Avant de lancer** (Kaggle) : *Settings* → *Accelerator* : **GPU T4 x2** (ou P100) · *Internet* : **On**.
Puis **Run All**. Les vidéos finies apparaissent dans `/kaggle/working/videos` (onglet *Output* → télécharger).

Modifie seulement la cellule **« Plans à rendre »** : chaque ligne = un plan (scène, durée, réglages).
Durée indicative sur T4 en 1080×1920 : hyperespace et planètes ≈ 5–15 s par image, galaxie et nébuleuses
≈ 20–60 s par image. Les 7 plans par défaut (≈ 18 s de vidéo) prennent donc quelques heures : lance d'abord
**APERCU = True** (≈ 15 min) pour vérifier les cadrages, puis le rendu final (une session Kaggle dure jusqu'à 12 h).

In [ ]:
import os, re, subprocess, sys, urllib.request, shutil, time
ROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.makedirs(f"{ROOT}/scenes", exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

def latest_blender(series="4.5"):
    for base in ("https://download.blender.org/release", "https://mirrors.ocf.berkeley.edu/blender/release",
                 "https://ftp.halifax.rwth-aachen.de/blender/release"):
        try:
            html = urllib.request.urlopen(f"{base}/Blender{series}/", timeout=30).read().decode()
            names = sorted(set(re.findall(rf"blender-{re.escape(series)}\.(\d+)-linux-x64\.tar\.xz", html)), key=int)
            if names:
                return f"{base}/Blender{series}/blender-{series}.{names[-1]}-linux-x64.tar.xz"
        except Exception as e:
            print("miroir indisponible :", base, e)
    return None

BLENDER = f"{ROOT}/blender/blender"
if not os.path.exists(BLENDER):
    url = latest_blender()
    print("Téléchargement :", url)
    subprocess.run(f"wget -q -O /tmp/blender.tar.xz '{url}' && mkdir -p {ROOT}/blender && "
                   f"tar -xf /tmp/blender.tar.xz -C {ROOT}/blender --strip-components=1 && rm /tmp/blender.tar.xz",
                   shell=True, check=True)
    subprocess.run("apt-get -qq update && apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 "
                   "libsm6 libxkbcommon0 ffmpeg > /dev/null", shell=True)
print(subprocess.run([BLENDER, "--version"], capture_output=True, text=True).stdout.splitlines()[0])

In [ ]:
%%writefile {ROOT}/scenes/commun.py
"""Outils communs aux scènes Blender (à lancer avec : blender -b -P scene.py -- [options]).

Réglages de rendu (Cycles sur GPU OptiX/CUDA si présent, sinon CPU), format TikTok 1080×1920 à 30 i/s,
étalonnage AgX contrasté, halo par le compositeur, ciel étoilé procédural, et petites aides pour les nœuds.
"""
import argparse
import math
import sys

import bpy

W, H, FPS = 1080, 1920, 30


def args(extra=None):
    """Options après « -- » : --out, --res (échelle %), --samples, --frames, --start, --engine, --seed."""
    argv = sys.argv[sys.argv.index("--") + 1:] if "--" in sys.argv else []
    p = argparse.ArgumentParser()
    p.add_argument("--out", default="//rendu/")
    p.add_argument("--res", type=int, default=100, help="échelle de résolution en % (25 pour un aperçu)")
    p.add_argument("--samples", type=int, default=96)
    p.add_argument("--frames", type=int, default=150, help="nombre d'images (30 = 1 s)")
    p.add_argument("--start", type=int, default=1)
    p.add_argument("--engine", default="CYCLES", choices=["CYCLES", "EEVEE"])
    p.add_argument("--seed", type=int, default=1)
    p.add_argument("--still", type=int, default=0, help="n'image unique à rendre (0 = toute l'animation)")
    p.add_argument("--debruitage", default="oidn", choices=["oidn", "optix", "aucun"],
                   help="oidn : OpenImageDenoise (marche partout) ; optix : plus rapide mais dépend du pilote")
    for a, kw in (extra or {}).items():
        p.add_argument(a, **kw)
    return p.parse_args(argv)


def reset():
    bpy.ops.wm.read_factory_settings(use_empty=True)
    return bpy.context.scene


def setup_render(sc, a, motion_blur=False):
    sc.render.resolution_x, sc.render.resolution_y = W, H
    sc.render.resolution_percentage = a.res
    sc.render.fps = FPS
    sc.frame_start = 1
    sc.frame_end = a.frames
    if a.engine == "EEVEE":
        sc.render.engine = "BLENDER_EEVEE_NEXT"
        sc.eevee.taa_render_samples = max(16, a.samples)
        sc.eevee.use_volumetric_shadows = True
        sc.eevee.volumetric_tile_size = "2"
        sc.eevee.volumetric_samples = 128
    else:
        sc.render.engine = "CYCLES"
        sc.cycles.samples = a.samples
        sc.cycles.use_denoising = True
        sc.cycles.volume_step_rate = 1.0
        sc.cycles.volume_max_steps = 512
        sc.cycles.max_bounces = 6
        sc.cycles.volume_bounces = 1
        sc.cycles.seed = a.seed
        sc.cycles.device = "CPU"
        try:                                               # GPU : OptiX (RTX, T4) puis CUDA (P100)
            prefs = bpy.context.preferences.addons["cycles"].preferences
            for kind in ("OPTIX", "CUDA"):
                try:
                    prefs.compute_device_type = kind
                    prefs.get_devices()
                    gpus = [d for d in prefs.devices if d.type == kind]
                    if gpus:
                        for d in prefs.devices:
                            d.use = d.type == kind
                        sc.cycles.device = "GPU"
                        print(f"[rendu] GPU {kind} : {[d.name for d in gpus]}")
                        break
                except TypeError:
                    continue
        except Exception as e:                             # pas de GPU : on rend sur le processeur
            print("[rendu] CPU", e)
        # débruitage : OptiX échoue sur certains pilotes (ex. Kaggle) → OpenImageDenoise par défaut, sur le CPU
        sc.cycles.use_denoising = a.debruitage != "aucun"
        sc.cycles.denoiser = "OPTIX" if a.debruitage == "optix" else "OPENIMAGEDENOISE"
        if hasattr(sc.cycles, "denoising_use_gpu"):
            sc.cycles.denoising_use_gpu = False
        print(f"[rendu] débruitage : {a.debruitage}")
    sc.render.use_motion_blur = motion_blur
    sc.render.motion_blur_shutter = 0.5
    sc.view_settings.view_transform = "AgX"
    try:
        sc.view_settings.look = "AgX - Punchy"
    except TypeError:
        pass
    sc.render.image_settings.file_format = "PNG"
    sc.render.image_settings.color_depth = "8"
    sc.render.filepath = a.out


def bloom(sc, size=7, mix=0.0, threshold=0.6, kind="FOG_GLOW"):
    """Halo des hautes lumières dans le compositeur."""
    sc.use_nodes = True
    nt = sc.node_tree
    nt.nodes.clear()
    rl = nt.nodes.new("CompositorNodeRLayers")
    gl = nt.nodes.new("CompositorNodeGlare")
    gl.glare_type = kind
    gl.quality = "HIGH"
    gl.size = size
    gl.mix = mix
    gl.threshold = threshold
    out = nt.nodes.new("CompositorNodeComposite")
    nt.links.new(rl.outputs["Image"], gl.inputs["Image"])
    nt.links.new(gl.outputs["Image"], out.inputs["Image"])


class Nodes:
    """Petite aide pour écrire des arbres de nœuds lisibles."""

    def __init__(self, tree):
        self.t = tree
        self.n = tree.nodes
        self.l = tree.links

    def new(self, kind, **props):
        node = self.n.new(kind)
        for k, v in props.items():
            if k in node.inputs.keys():
                node.inputs[k].default_value = v
            else:
                setattr(node, k, v)
        return node

    def link(self, a, b):
        self.l.new(a, b)

    def math(self, op, a, b=None, clamp=False):
        m = self.n.new("ShaderNodeMath")
        m.operation = op
        m.use_clamp = clamp
        for i, v in enumerate((a, b)):
            if v is None:
                continue
            if isinstance(v, (int, float)):
                m.inputs[i].default_value = v
            else:
                self.l.new(v, m.inputs[i])
        return m.outputs[0]

    def ramp(self, src, stops, interp="LINEAR"):
        """stops : [(position, (r, g, b, a)), …]"""
        r = self.n.new("ShaderNodeValToRGB")
        r.color_ramp.interpolation = interp
        els = r.color_ramp.elements
        while len(els) > 1:
            els.remove(els[-1])
        els[0].position, els[0].color = stops[0]
        for pos, col in stops[1:]:
            e = els.new(pos)
            e.color = col
        self.l.new(src, r.inputs["Fac"])
        return r


def material(name):
    m = bpy.data.materials.new(name)
    m.use_nodes = True
    m.node_tree.nodes.clear()
    return m, Nodes(m.node_tree)


def starfield(sc, density=1.0, milky=0.25, tint=(0.5, 0.6, 1.0), strength=1.0):
    """Ciel : deux couches d'étoiles (Voronoi) de tailles et couleurs variées + voile de voie lactée."""
    w = bpy.data.worlds.new("ciel")
    sc.world = w
    w.use_nodes = True
    nt = w.node_tree
    nt.nodes.clear()
    N = Nodes(nt)
    tc = N.new("ShaderNodeTexCoord")
    total = None
    for scale, thr, gain in ((420.0, 0.045, 6.0), (130.0, 0.03, 18.0)):
        vo = N.new("ShaderNodeTexVoronoi", voronoi_dimensions="3D", feature="F1")
        vo.inputs["Scale"].default_value = scale * density
        vo.inputs["Randomness"].default_value = 1.0
        N.link(tc.outputs["Generated"], vo.inputs["Vector"])
        star = N.math("LESS_THAN", vo.outputs["Distance"], thr)
        sepc = N.new("ShaderNodeSeparateColor")
        N.link(vo.outputs["Color"], sepc.inputs["Color"])
        b = N.math("POWER", sepc.outputs["Red"], 6.0)
        lay = N.math("MULTIPLY", star, N.math("MULTIPLY", b, gain))
        total = lay if total is None else N.math("ADD", total, lay)
    no = N.new("ShaderNodeTexNoise", noise_dimensions="3D")
    no.inputs["Scale"].default_value = 2.0
    no.inputs["Detail"].default_value = 10.0
    no.inputs["Roughness"].default_value = 0.62
    N.link(tc.outputs["Generated"], no.inputs["Vector"])
    haze = N.math("MULTIPLY", N.math("POWER", no.outputs["Fac"], 4.0), milky)
    col = N.new("ShaderNodeMix", data_type="RGBA")
    col.inputs["Factor"].default_value = 1.0
    bg = N.new("ShaderNodeBackground")
    em_col = N.new("ShaderNodeCombineColor")
    lum = N.math("ADD", total, haze)
    N.link(N.math("MULTIPLY", lum, tint[0]), em_col.inputs["Red"])
    N.link(N.math("MULTIPLY", lum, tint[1]), em_col.inputs["Green"])
    N.link(N.math("MULTIPLY", lum, tint[2]), em_col.inputs["Blue"])
    N.link(em_col.outputs["Color"], bg.inputs["Color"])
    bg.inputs["Strength"].default_value = strength
    out = N.new("ShaderNodeOutputWorld")
    N.link(bg.outputs["Background"], out.inputs["Surface"])
    for nd in list(nt.nodes):                              # nettoie les nœuds orphelins
        if not any(o.is_linked for o in nd.outputs) and nd.type not in ("OUTPUT_WORLD",):
            nt.nodes.remove(nd)
    return w


def camera(sc, lens=24.0, dof=None):
    cd = bpy.data.cameras.new("cam")
    cd.lens = lens
    cd.clip_start = 0.01
    cd.clip_end = 5000
    if dof:
        cd.dof.use_dof = True
        cd.dof.focus_distance, cd.dof.aperture_fstop = dof
    cam = bpy.data.objects.new("cam", cd)
    sc.collection.objects.link(cam)
    sc.camera = cam
    return cam


def look_at(obj, target):
    from mathutils import Vector
    d = Vector(target) - obj.location
    obj.rotation_euler = d.to_track_quat("-Z", "Y").to_euler()


def key_path(cam, frames, points, targets, ease=True):
    """Trajectoire de caméra : positions et cibles clés réparties sur l'animation (courbes lissées)."""
    n = len(points)
    for i, (p, tg) in enumerate(zip(points, targets)):
        f = 1 + round((frames - 1) * i / max(1, n - 1))
        cam.location = p
        look_at(cam, tg)
        cam.keyframe_insert("location", frame=f)
        cam.keyframe_insert("rotation_euler", frame=f)
    if cam.animation_data and cam.animation_data.action:
        for fc in _fcurves(cam.animation_data.action):
            for kp in fc.keyframe_points:
                kp.interpolation = "BEZIER"
                kp.easing = "AUTO" if ease else "EASE_IN_OUT"


def _fcurves(action):
    if hasattr(action, "fcurves") and len(action.fcurves):
        return action.fcurves
    out = []                                               # Blender 4.4+ : actions en couches
    for layer in getattr(action, "layers", []):
        for strip in layer.strips:
            for bag in strip.channelbags:
                out.extend(bag.fcurves)
    return out


def key_value(socket, frames, v0, v1):
    socket.default_value = v0
    socket.keyframe_insert("default_value", frame=1)
    socket.default_value = v1
    socket.keyframe_insert("default_value", frame=frames)


def linear_all():
    for a in bpy.data.actions:
        for fc in _fcurves(a):
            for kp in fc.keyframe_points:
                kp.interpolation = "LINEAR"


def render(sc, a):
    if a.still:
        sc.frame_set(a.still)
        sc.render.filepath = a.out if a.out.endswith(".png") else a.out + f"{a.still:04d}.png"
        bpy.ops.render.render(write_still=True)
    else:
        bpy.ops.render.render(animation=True)

In [ ]:
%%writefile {ROOT}/scenes/nebuleuse.py
"""Nébuleuse volumétrique : la caméra avance lentement entre des piliers de gaz éclairés à contre-jour.

blender -b -P nebuleuse.py -- --out /chemin/neb_ --palette bleu --frames 150
Palettes : bleu (bleu/violet/magenta), feu (rouge/orange/or), emeraude (vert/turquoise/or), rose (magenta/rouge).
"""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import bpy  # noqa: E402

import commun as C  # noqa: E402

PALETTES = {
    "bleu": [(0.0, (0.01, 0.03, 0.25, 1)), (0.35, (0.10, 0.20, 1.0, 1)), (0.6, (0.55, 0.20, 1.0, 1)),
             (0.8, (1.0, 0.20, 0.55, 1)), (1.0, (1.0, 0.65, 0.40, 1))],
    "feu": [(0.0, (0.15, 0.01, 0.0, 1)), (0.35, (0.9, 0.08, 0.02, 1)), (0.6, (1.0, 0.30, 0.03, 1)),
            (0.85, (1.0, 0.65, 0.15, 1)), (1.0, (1.0, 0.95, 0.7, 1))],
    "emeraude": [(0.0, (0.0, 0.05, 0.04, 1)), (0.35, (0.0, 0.5, 0.35, 1)), (0.6, (0.1, 0.9, 0.6, 1)),
                 (0.8, (0.9, 0.85, 0.2, 1)), (1.0, (1.0, 0.95, 0.8, 1))],
    "rose": [(0.0, (0.1, 0.0, 0.08, 1)), (0.35, (0.8, 0.05, 0.3, 1)), (0.6, (1.0, 0.15, 0.55, 1)),
             (0.8, (0.6, 0.3, 1.0, 1)), (1.0, (1.0, 0.85, 0.95, 1))],
}


def nebula_material(palette, seed, dom):
    m, N = C.material("nebuleuse")
    tc = N.new("ShaderNodeTexCoord")
    tc.object = dom                                        # coordonnées du domaine, même vues depuis un autre objet
    # déformation du domaine : de grandes volutes
    warp = N.new("ShaderNodeTexNoise", noise_dimensions="4D")
    warp.inputs["Scale"].default_value = 0.35
    warp.inputs["Detail"].default_value = 3.0
    warp.inputs["W"].default_value = seed * 3.1
    N.link(tc.outputs["Object"], warp.inputs["Vector"])
    wv = N.new("ShaderNodeVectorMath", operation="SUBTRACT")
    N.link(warp.outputs["Color"], wv.inputs[0])
    wv.inputs[1].default_value = (0.5, 0.5, 0.5)
    ws = N.new("ShaderNodeVectorMath", operation="SCALE")
    N.link(wv.outputs["Vector"], ws.inputs[0])
    ws.inputs["Scale"].default_value = 2.2
    q = N.new("ShaderNodeVectorMath", operation="ADD")
    N.link(tc.outputs["Object"], q.inputs[0])
    N.link(ws.outputs["Vector"], q.inputs[1])
    # forme : de grands volumes (bruit à basse fréquence, très contrasté)
    big = N.new("ShaderNodeTexNoise", noise_dimensions="4D")
    big.inputs["Scale"].default_value = 0.22
    big.inputs["Detail"].default_value = 2.0
    big.inputs["W"].default_value = seed * 7.7
    N.link(q.outputs["Vector"], big.inputs["Vector"])
    # détail : volutes fines et filaments
    det = N.new("ShaderNodeTexNoise", noise_dimensions="4D")
    det.inputs["Scale"].default_value = 1.1
    det.inputs["Detail"].default_value = 15.0
    det.inputs["Roughness"].default_value = 0.68
    det.inputs["Distortion"].default_value = 0.35
    det.inputs["W"].default_value = seed * 1.3
    N.link(q.outputs["Vector"], det.inputs["Vector"])
    rid = N.math("ABSOLUTE", N.math("SUBTRACT", det.outputs["Fac"], 0.5))
    ridge = N.math("POWER", N.math("SUBTRACT", 1.0, N.math("MULTIPLY", rid, 2.0)), 3.0)
    n = N.math("ADD", N.math("MULTIPLY", big.outputs["Fac"], 1.0),
               N.math("ADD", N.math("MULTIPLY", det.outputs["Fac"], 0.55), N.math("MULTIPLY", ridge, 0.12)))
    n_raw = n
    map_n = N.new("ShaderNodeMapRange", clamp=True)        # bornes réglées par calibrate()
    N.link(n_raw, map_n.inputs["Value"])
    dens_r = N.ramp(map_n.outputs["Result"], [(0.0, (0, 0, 0, 1)), (0.35, (0.25, 0.25, 0.25, 1)),
                                              (1.0, (1, 1, 1, 1))])
    dens = N.math("MULTIPLY", dens_r.outputs["Color"], 0.9)
    # couleur : champ « d'ionisation » lent, indépendant de la densité
    ion = N.new("ShaderNodeTexNoise", noise_dimensions="4D")
    ion.inputs["Scale"].default_value = 0.5
    ion.inputs["Detail"].default_value = 2.0
    ion.inputs["W"].default_value = seed * 5.3 + 2
    N.link(q.outputs["Vector"], ion.inputs["Vector"])
    ionr = N.math("ADD", N.math("MULTIPLY", ion.outputs["Fac"], 4.0), N.math("MULTIPLY", det.outputs["Fac"], 1.2))
    map_i = N.new("ShaderNodeMapRange", clamp=True)
    N.link(ionr, map_i.inputs["Value"])
    colr = N.ramp(map_i.outputs["Result"], PALETTES[palette])
    albedo = N.new("ShaderNodeMix", data_type="RGBA")      # le gaz diffuse sa propre couleur (pas du blanc)
    albedo.inputs["Factor"].default_value = 0.35
    N.link(colr.outputs["Color"], albedo.inputs["A"])
    albedo.inputs["B"].default_value = (0.02, 0.02, 0.03, 1)
    # le gaz fin luit, le gaz épais absorbe (poussière sombre)
    thin = N.math("SUBTRACT", 1.0, dens_r.outputs["Color"])
    glow = N.math("MULTIPLY", N.math("ADD", thin, 0.12), N.math("MULTIPLY", dens, 3.5))
    vol = N.new("ShaderNodeVolumePrincipled")
    N.link(dens, vol.inputs["Density"])
    N.link(albedo.outputs["Result"], vol.inputs["Color"])
    N.link(colr.outputs["Color"], vol.inputs["Emission Color"])
    N.link(glow, vol.inputs["Emission Strength"])
    vol.inputs["Absorption Color"].default_value = (0.05, 0.03, 0.04, 1)
    vol.inputs["Anisotropy"].default_value = 0.45            # diffusion vers l'avant : contre-jour lumineux
    out = N.new("ShaderNodeOutputMaterial")
    N.link(vol.outputs["Volume"], out.inputs["Volume"])
    return m, (warp, big, det, ion), {"n": (n_raw, map_n), "ion": (ionr, map_i), "out": out, "vol": vol}


def calibrate(sc, dom, m, h, coverage=0.32):
    """Mesure la distribution réelle du bruit sur trois coupes du domaine (rendu minuscule en EXR), puis règle
    les bornes : le gaz occupe ≈ `coverage` de l'espace, la palette couvre toute la plage de couleurs."""
    import numpy as np
    nt = m.node_tree
    em = nt.nodes.new("ShaderNodeEmission")
    out, vol = h["out"], h["vol"]
    for l in list(out.inputs["Volume"].links):
        nt.links.remove(l)
    nt.links.new(em.outputs[0], out.inputs["Surface"])
    saved = (sc.render.resolution_x, sc.render.resolution_y, sc.render.resolution_percentage, sc.camera,
             sc.render.use_compositing, sc.render.image_settings.file_format, sc.render.filepath,
             sc.cycles.samples, sc.cycles.use_denoising, sc.render.engine)
    sc.render.engine = "CYCLES"
    sc.render.resolution_x, sc.render.resolution_y, sc.render.resolution_percentage = 96, 96, 100
    sc.render.use_compositing = False
    sc.render.image_settings.file_format = "OPEN_EXR"
    sc.cycles.samples, sc.cycles.use_denoising = 1, False
    dom.hide_render = True
    cd = bpy.data.cameras.new("calib")
    cd.type = "ORTHO"
    cd.ortho_scale = 2 * max(dom.scale.x, dom.scale.y)
    cam = bpy.data.objects.new("calib", cd)
    sc.collection.objects.link(cam)
    sc.camera = cam
    bpy.ops.mesh.primitive_plane_add(size=2)
    pl = bpy.context.object
    pl.scale = (dom.scale.x, dom.scale.y, 1)
    pl.data.materials.append(m)
    res = {}
    import tempfile
    path = os.path.join(tempfile.mkdtemp(), "calib.exr")
    for key in ("n", "ion"):
        nt.links.new(h[key][0], em.inputs["Strength"])
        vals = []
        for z in (-0.6, 0.0, 0.6):
            pl.location = (0, 0, z * dom.scale.z)
            cam.location = (0, 0, z * dom.scale.z + 3)
            cam.rotation_euler = (0, 0, 0)
            sc.render.filepath = path
            bpy.ops.render.render(write_still=True)
            img = bpy.data.images.load(path)
            px = np.array(img.pixels[:]).reshape(-1, 4)[:, 0]
            bpy.data.images.remove(img)
            vals.append(px)
        res[key] = np.concatenate(vals)
    hn, hi = h["n"][1], h["ion"][1]
    hn.inputs["From Min"].default_value = float(np.quantile(res["n"], 1 - coverage))
    hn.inputs["From Max"].default_value = float(np.quantile(res["n"], 1 - coverage * 0.12))
    hi.inputs["From Min"].default_value = float(np.quantile(res["ion"], 0.04))
    hi.inputs["From Max"].default_value = float(np.quantile(res["ion"], 0.97))
    print("[calibrage] gaz", hn.inputs["From Min"].default_value, hn.inputs["From Max"].default_value,
          "couleur", hi.inputs["From Min"].default_value, hi.inputs["From Max"].default_value)
    bpy.data.objects.remove(pl)
    bpy.data.objects.remove(cam)
    nt.nodes.remove(em)
    nt.links.new(vol.outputs["Volume"], out.inputs["Volume"])
    dom.hide_render = False
    (sc.render.resolution_x, sc.render.resolution_y, sc.render.resolution_percentage, sc.camera,
     sc.render.use_compositing, sc.render.image_settings.file_format, sc.render.filepath,
     sc.cycles.samples, sc.cycles.use_denoising, sc.render.engine) = saved


def main():
    a = C.args({"--palette": dict(default="bleu", choices=list(PALETTES)),
                "--evolution": dict(type=float, default=0.25, help="vitesse d'évolution des volutes"),
                "--couverture": dict(type=float, default=0.2, help="part de l'espace remplie de gaz")})
    sc = C.reset()
    C.setup_render(sc, a)
    C.bloom(sc, size=8, threshold=0.5)
    C.starfield(sc, density=1.0, milky=0.12, strength=1.0)
    bpy.ops.mesh.primitive_cube_add(size=2)
    dom = bpy.context.object
    dom.name = "gaz"
    dom.scale = (9, 9, 16)
    m, noises, handles = nebula_material(a.palette, a.seed, dom)
    dom.data.materials.append(m)
    calibrate(sc, dom, m, handles, a.couverture)
    for nd in noises:                                     # le gaz ondule lentement
        C.key_value(nd.inputs["W"], a.frames, nd.inputs["W"].default_value,
                    nd.inputs["W"].default_value + a.evolution)
    # étoiles enfouies : elles illuminent le gaz par derrière
    pal = PALETTES[a.palette]
    for (x, y, z), power, col in (((1.5, 2.0, -6.0), 5000, pal[-1][1][:3]), ((-3.0, -1.0, -11.0), 3500, (0.8, 0.85, 1.0)),
                                  ((2.5, -3.0, 3.0), 900, pal[-2][1][:3])):
        ld = bpy.data.lights.new("etoile", "POINT")
        ld.energy = power
        ld.color = col
        ld.shadow_soft_size = 0.3
        lo = bpy.data.objects.new("etoile", ld)
        lo.location = (x, y, z)
        sc.collection.objects.link(lo)
        bpy.ops.mesh.primitive_uv_sphere_add(radius=0.06, location=(x, y, z))
        s = bpy.context.object
        sm, SN = C.material("coeur")
        em = SN.new("ShaderNodeEmission")
        em.inputs["Color"].default_value = (*col, 1)
        em.inputs["Strength"].default_value = 400
        o = SN.new("ShaderNodeOutputMaterial")
        SN.link(em.outputs["Emission"], o.inputs["Surface"])
        s.data.materials.append(sm)
    cam = C.camera(sc, lens=22)
    C.key_path(cam, a.frames, [(0.3, -0.6, 13.0), (-0.2, 0.2, 9.0)],
               [(0.8, 0.8, -4.0), (0.3, 1.0, -8.0)])
    C.render(sc, a)


main()

In [ ]:
%%writefile {ROOT}/scenes/planete.py
"""Planète photoréaliste : bandes nuageuses, atmosphère lumineuse sur le limbe, anneaux à lacunes avec l'ombre de
la planète, lune, ciel étoilé. La caméra glisse lentement : les anneaux balaient l'image.

blender -b -P planete.py -- --out /chemin/pla_ --type gazeuse --frames 150
Types : gazeuse (Saturne/Jupiter), glace (bleu-vert pâle), rose (exotique), rocheuse (sans anneaux par défaut).
"""
import math
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import bpy  # noqa: E402

import commun as C  # noqa: E402

TYPES = {
    "gazeuse": dict(cols=[(0.0, (0.35, 0.22, 0.12, 1)), (0.3, (0.85, 0.68, 0.45, 1)), (0.55, (0.95, 0.88, 0.72, 1)),
                          (0.75, (0.7, 0.45, 0.25, 1)), (1.0, (0.98, 0.92, 0.8, 1))],
                    atmo=(1.0, 0.8, 0.55), ring=(0.95, 0.88, 0.75), anneaux=True, bandes=9.0),
    "glace": dict(cols=[(0.0, (0.35, 0.75, 0.8, 1)), (0.5, (0.55, 0.9, 0.9, 1)), (1.0, (0.8, 0.98, 0.98, 1))],
                  atmo=(0.45, 0.9, 1.0), ring=(0.6, 0.9, 0.85), anneaux=True, bandes=4.0),
    "rose": dict(cols=[(0.0, (0.55, 0.25, 0.5, 1)), (0.4, (0.9, 0.65, 0.85, 1)), (0.7, (0.98, 0.85, 0.95, 1)),
                       (1.0, (0.7, 0.4, 0.7, 1))],
                 atmo=(1.0, 0.6, 0.9), ring=(1.0, 0.45, 0.4), anneaux=True, bandes=6.0),
    "rocheuse": dict(cols=[(0.0, (0.25, 0.12, 0.08, 1)), (0.45, (0.55, 0.3, 0.2, 1)), (0.7, (0.75, 0.55, 0.45, 1)),
                           (1.0, (0.9, 0.85, 0.8, 1))],
                     atmo=(1.0, 0.55, 0.4), ring=(0.8, 0.7, 0.6), anneaux=False, bandes=0.0),
}


def planet_material(kind, seed):
    T = TYPES[kind]
    m, N = C.material("planete")
    tc = N.new("ShaderNodeTexCoord")
    warp = N.new("ShaderNodeTexNoise", noise_dimensions="4D", **{"W": seed * 2.3})
    warp.inputs["Scale"].default_value = 2.5
    warp.inputs["Detail"].default_value = 8.0
    warp.inputs["Roughness"].default_value = 0.6
    N.link(tc.outputs["Object"], warp.inputs["Vector"])
    if T["bandes"] > 0:                                    # géante gazeuse : bandes de latitude tourmentées
        sep = N.new("ShaderNodeSeparateXYZ")
        N.link(tc.outputs["Object"], sep.inputs["Vector"])
        lat = N.math("ADD", sep.outputs["Z"], N.math("MULTIPLY", N.math("SUBTRACT", warp.outputs["Fac"], 0.5), 0.12))
        band = N.new("ShaderNodeTexWave", wave_type="BANDS", bands_direction="Z", wave_profile="SIN")
        band.inputs["Scale"].default_value = T["bandes"] * 0.38
        band.inputs["Distortion"].default_value = 7.0
        band.inputs["Detail"].default_value = 6.0
        band.inputs["Detail Scale"].default_value = 2.0
        comb = N.new("ShaderNodeCombineXYZ")
        N.link(lat, comb.inputs["Z"])
        N.link(N.math("MULTIPLY", sep.outputs["X"], 0.2), comb.inputs["X"])
        N.link(comb.outputs["Vector"], band.inputs["Vector"])
        fine = N.new("ShaderNodeTexNoise", noise_dimensions="3D")
        fine.inputs["Scale"].default_value = 18.0
        fine.inputs["Detail"].default_value = 10.0
        N.link(tc.outputs["Object"], fine.inputs["Vector"])
        v = N.math("ADD", N.math("MULTIPLY", band.outputs["Fac"], 0.8), N.math("MULTIPLY", fine.outputs["Fac"], 0.25))
    else:                                                  # planète rocheuse : continents, cratères, calottes
        v = N.math("ADD", warp.outputs["Fac"], 0.0)
    col = N.ramp(v, T["cols"])
    bs = N.new("ShaderNodeBsdfPrincipled")
    N.link(col.outputs["Color"], bs.inputs["Base Color"])
    bs.inputs["Roughness"].default_value = 0.85
    if T["bandes"] == 0:
        bump = N.new("ShaderNodeBump")
        bump.inputs["Strength"].default_value = 0.4
        N.link(warp.outputs["Fac"], bump.inputs["Height"])
        N.link(bump.outputs["Normal"], bs.inputs["Normal"])
    out = N.new("ShaderNodeOutputMaterial")
    N.link(bs.outputs["BSDF"], out.inputs["Surface"])
    return m


def atmosphere_material(color, sun_dir):
    """Liseré lumineux : émission sur les bords (Fresnel), seulement du côté éclairé, par-dessus la planète."""
    m, N = C.material("atmosphere")
    lw = N.new("ShaderNodeLayerWeight")
    lw.inputs["Blend"].default_value = 0.35
    geo = N.new("ShaderNodeNewGeometry")
    dot = N.new("ShaderNodeVectorMath", operation="DOT_PRODUCT")
    N.link(geo.outputs["Normal"], dot.inputs[0])
    dot.inputs[1].default_value = sun_dir
    lit = N.new("ShaderNodeMapRange", clamp=True)
    N.link(dot.outputs["Value"], lit.inputs["Value"])
    lit.inputs["From Min"].default_value = -0.08
    lit.inputs["From Max"].default_value = 0.6
    rim = N.math("POWER", lw.outputs["Facing"], 2.2)
    k = N.math("MULTIPLY", rim, lit.outputs["Result"])
    em = N.new("ShaderNodeEmission")
    em.inputs["Color"].default_value = (*color, 1)
    N.link(N.math("MULTIPLY", k, 6.0), em.inputs["Strength"])
    tr = N.new("ShaderNodeBsdfTransparent")
    add = N.new("ShaderNodeAddShader")
    N.link(em.outputs["Emission"], add.inputs[0])
    N.link(tr.outputs["BSDF"], add.inputs[1])
    out = N.new("ShaderNodeOutputMaterial")
    N.link(add.outputs["Shader"], out.inputs["Surface"])
    return m


def ring_material(color, seed):
    m, N = C.material("anneaux")
    tc = N.new("ShaderNodeTexCoord")
    ln = N.new("ShaderNodeVectorMath", operation="LENGTH")
    N.link(tc.outputs["Object"], ln.inputs[0])
    r = ln.outputs["Value"]                                # rayon (anneaux de 1,35 à 2,35 rayons planétaires)
    w1 = N.new("ShaderNodeTexWave", wave_type="RINGS", rings_direction="SPHERICAL", wave_profile="SIN")
    w1.inputs["Scale"].default_value = 30.0
    w1.inputs["Distortion"].default_value = 1.5
    w1.inputs["Detail"].default_value = 8.0
    N.link(tc.outputs["Object"], w1.inputs["Vector"])
    w2 = N.new("ShaderNodeTexWave", wave_type="RINGS", rings_direction="SPHERICAL", wave_profile="SAW")
    w2.inputs["Scale"].default_value = 7.0
    w2.inputs["Distortion"].default_value = 0.6
    w2.inputs["Phase Offset"].default_value = seed * 1.7
    N.link(tc.outputs["Object"], w2.inputs["Vector"])
    prof = N.ramp(N.math("SUBTRACT", N.math("DIVIDE", r, 2.4), 0.0),
                  [(0.0, (0, 0, 0, 1)), (0.55, (0, 0, 0, 1)), (0.57, (0.5, 0.5, 0.5, 1)), (0.72, (1, 1, 1, 1)),
                   (0.76, (0.08, 0.08, 0.08, 1)), (0.78, (0.9, 0.9, 0.9, 1)), (0.93, (0.6, 0.6, 0.6, 1)),
                   (0.98, (0, 0, 0, 1))])          # lacune type « division de Cassini » vers 0,77
    a = N.math("MULTIPLY", prof.outputs["Color"],
               N.math("ADD", N.math("MULTIPLY", w1.outputs["Fac"], 0.55), N.math("MULTIPLY", w2.outputs["Fac"], 0.45)))
    a = N.math("MULTIPLY", a, 1.3, clamp=True)
    tint = N.new("ShaderNodeMix", data_type="RGBA")
    tint.inputs["A"].default_value = (*color, 1)
    tint.inputs["B"].default_value = (color[0] * 0.55, color[1] * 0.5, color[2] * 0.45, 1)
    N.link(w2.outputs["Fac"], tint.inputs["Factor"])
    bs = N.new("ShaderNodeBsdfPrincipled")
    N.link(tint.outputs["Result"], bs.inputs["Base Color"])
    bs.inputs["Roughness"].default_value = 0.9
    N.link(a, bs.inputs["Alpha"])
    out = N.new("ShaderNodeOutputMaterial")
    N.link(bs.outputs["BSDF"], out.inputs["Surface"])
    return m


def emissive(name, color, strength):
    m, N = C.material(name)
    em = N.new("ShaderNodeEmission")
    em.inputs["Color"].default_value = (*color, 1)
    em.inputs["Strength"].default_value = strength
    out = N.new("ShaderNodeOutputMaterial")
    N.link(em.outputs["Emission"], out.inputs["Surface"])
    return m


def main():
    a = C.args({"--type": dict(default="gazeuse", choices=list(TYPES)),
                "--inclinaison": dict(type=float, default=24.0, help="inclinaison des anneaux (degrés)"),
                "--lune": dict(type=int, default=1)})
    T = TYPES[a.type]
    sc = C.reset()
    C.setup_render(sc, a)
    C.bloom(sc, size=7, threshold=0.8)
    C.starfield(sc, density=1.0, milky=0.15, strength=1.0)
    sun_dir = (0.92, -0.25, 0.3)                          # lumière de côté : moitié éclairée, terminateur net
    ld = bpy.data.lights.new("soleil", "SUN")
    ld.energy = 5.0
    ld.angle = math.radians(0.5)
    ld.color = (1.0, 0.97, 0.92)
    lo = bpy.data.objects.new("soleil", ld)
    sc.collection.objects.link(lo)
    C.look_at(lo, (-sun_dir[0], -sun_dir[1], -sun_dir[2]))
    root = bpy.data.objects.new("systeme", None)
    sc.collection.objects.link(root)
    root.rotation_euler = (math.radians(a.inclinaison), math.radians(-12), 0)
    bpy.ops.mesh.primitive_uv_sphere_add(segments=192, ring_count=96, radius=1.0)
    pl = bpy.context.object
    bpy.ops.object.shade_smooth()
    pl.data.materials.append(planet_material(a.type, a.seed))
    pl.parent = root
    bpy.ops.mesh.primitive_uv_sphere_add(segments=128, ring_count=64, radius=1.035)
    at = bpy.context.object
    bpy.ops.object.shade_smooth()
    at.data.materials.append(atmosphere_material(T["atmo"], sun_dir))
    at.parent = root
    if T["anneaux"]:
        bpy.ops.mesh.primitive_circle_add(vertices=256, radius=2.4, fill_type="NGON")
        rg = bpy.context.object
        rg.data.materials.append(ring_material(T["ring"], a.seed))
        rg.parent = root
    if a.lune:
        bpy.ops.mesh.primitive_uv_sphere_add(segments=64, ring_count=32, radius=0.12, location=(-3.2, 2.5, 0.9))
        mo = bpy.context.object
        bpy.ops.object.shade_smooth()
        mm, MN = C.material("lune")
        no = MN.new("ShaderNodeTexNoise", noise_dimensions="3D")
        no.inputs["Scale"].default_value = 6.0
        no.inputs["Detail"].default_value = 12.0
        bs = MN.new("ShaderNodeBsdfPrincipled")
        MN.link(MN.ramp(no.outputs["Fac"], [(0.3, (0.25, 0.24, 0.23, 1)), (0.7, (0.7, 0.68, 0.65, 1))]).outputs["Color"],
                bs.inputs["Base Color"])
        o = MN.new("ShaderNodeOutputMaterial")
        MN.link(bs.outputs["BSDF"], o.inputs["Surface"])
        mo.data.materials.append(mm)
    cam = C.camera(sc, lens=42)
    pts, tgs = [], []
    for u in (0.0, 0.5, 1.0):                              # travelling en arc : les anneaux tournent dans l'image
        ang = math.radians(-35 + 30 * u)
        d = 6.4 - 0.9 * u
        pts.append((d * math.sin(ang), -d * math.cos(ang), 0.9 - 0.5 * u))
        tgs.append((0.25 - 0.2 * u, 0, -0.35))
    C.key_path(cam, a.frames, pts, tgs)
    C.render(sc, a)


main()

In [ ]:
%%writefile {ROOT}/scenes/hyperespace.py
"""Vol à travers les étoiles : des milliers d'étoiles en vrais points 3D, la caméra accélère, le flou de
mouvement les étire en traits (comme l'ouverture de la vidéo de référence). Un voile de nébuleuse au fond.

blender -b -P hyperespace.py -- --out /chemin/hyp_ --frames 120 --vitesse 1.0
"""
import math
import os
import random
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import bpy  # noqa: E402

import commun as C  # noqa: E402


def point_cloud(name, pts, radius, color, strength):
    me = bpy.data.meshes.new(name)
    me.from_pydata(pts, [], [])
    ob = bpy.data.objects.new(name, me)
    bpy.context.scene.collection.objects.link(ob)
    m, N = C.material(name)
    em = N.new("ShaderNodeEmission")
    em.inputs["Color"].default_value = (*color, 1)
    em.inputs["Strength"].default_value = strength
    out = N.new("ShaderNodeOutputMaterial")
    N.link(em.outputs["Emission"], out.inputs["Surface"])
    gn = bpy.data.node_groups.new(name, "GeometryNodeTree")
    gn.interface.new_socket("Geometry", in_out="INPUT", socket_type="NodeSocketGeometry")
    gn.interface.new_socket("Geometry", in_out="OUTPUT", socket_type="NodeSocketGeometry")
    gi = gn.nodes.new("NodeGroupInput")
    go = gn.nodes.new("NodeGroupOutput")
    mp = gn.nodes.new("GeometryNodeMeshToPoints")
    mp.inputs["Radius"].default_value = radius
    sm = gn.nodes.new("GeometryNodeSetMaterial")
    sm.inputs["Material"].default_value = m
    gn.links.new(gi.outputs[0], mp.inputs["Mesh"])
    gn.links.new(mp.outputs["Points"], sm.inputs["Geometry"])
    gn.links.new(sm.outputs["Geometry"], go.inputs[0])
    mod = ob.modifiers.new(name, "NODES")
    mod.node_group = gn
    return ob


def main():
    a = C.args({"--vitesse": dict(type=float, default=1.0), "--etoiles": dict(type=int, default=7000)})
    sc = C.reset()
    C.setup_render(sc, a, motion_blur=True)
    sc.render.motion_blur_shutter = 0.6
    C.bloom(sc, size=6, threshold=0.7)
    C.starfield(sc, density=1.0, milky=0.35, tint=(0.45, 0.4, 1.0), strength=0.8)
    rng = random.Random(a.seed)
    length = 160.0 * a.vitesse
    groups = {"blanc": ((1.0, 1.0, 1.0), []), "bleu": ((0.55, 0.7, 1.0), []), "or": ((1.0, 0.75, 0.4), [])}
    for _ in range(a.etoiles):
        r = 1.5 + abs(rng.gauss(0, 7.0))                   # un couloir vide au centre : on fonce entre les étoiles
        th = rng.uniform(0, 2 * math.pi)
        z = -rng.uniform(0, length + 60)
        key = rng.choices(list(groups), weights=(6, 3, 2))[0]
        groups[key][1].append((r * math.cos(th), r * math.sin(th), z))
    for name, (col, pts) in groups.items():
        point_cloud(name, pts, 0.018, col, 30.0)
    cam = C.camera(sc, lens=20)
    cam.rotation_euler = (0, 0, 0)                        # regarde vers −z
    # accélération progressive (ease-in), puis plein régime
    n = a.frames
    for f in range(1, n + 1, max(1, n // 12)):
        u = (f - 1) / max(1, n - 1)
        cam.location = (0, 0, -length * (u ** 2.2))
        cam.keyframe_insert("location", frame=f)
    cam.location = (0, 0, -length)
    cam.keyframe_insert("location", frame=n)
    cam.rotation_euler = (0, 0, 0)
    cam.keyframe_insert("rotation_euler", frame=1)
    cam.rotation_euler = (0, 0, math.radians(25))          # roulis lent
    cam.keyframe_insert("rotation_euler", frame=n)
    C.render(sc, a)


main()

In [ ]:
%%writefile {ROOT}/scenes/galaxie.py
"""Galaxie spirale volumétrique : bras de gaz bleu, bulbe doré, régions roses de formation d'étoiles, couloirs
de poussière sombre. La caméra plonge depuis le dessus vers une vue rasante pendant que la galaxie tourne.

blender -b -P galaxie.py -- --out /chemin/gal_ --frames 150 --bras 2
"""
import math
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import bpy  # noqa: E402

import commun as C  # noqa: E402


def galaxy_material(arms, twist, seed):
    m, N = C.material("galaxie")
    tc = N.new("ShaderNodeTexCoord")
    sep = N.new("ShaderNodeSeparateXYZ")
    N.link(tc.outputs["Object"], sep.inputs["Vector"])
    x, y, z = sep.outputs["X"], sep.outputs["Y"], sep.outputs["Z"]
    r = N.math("SQRT", N.math("ADD", N.math("MULTIPLY", x, x), N.math("MULTIPLY", y, y)))
    th = N.math("ARCTAN2", y, x)
    # spirale logarithmique : phase = bras·(θ − torsion·ln r)
    ph = N.math("MULTIPLY", N.math("SUBTRACT", th, N.math("MULTIPLY", N.math("LOGARITHM", N.math("ADD", r, 0.02),
                                                                                   2.718281828), twist)), arms)
    armw = N.math("POWER", N.math("MULTIPLY", N.math("ADD", N.math("COSINE", ph), 1.0), 0.5), 5.0)
    warp = N.new("ShaderNodeTexNoise", noise_dimensions="4D")
    warp.inputs["Scale"].default_value = 4.0
    warp.inputs["Detail"].default_value = 12.0
    warp.inputs["Roughness"].default_value = 0.62
    warp.inputs["W"].default_value = seed
    N.link(tc.outputs["Object"], warp.inputs["Vector"])
    clump = N.math("POWER", warp.outputs["Fac"], 3.5)
    disk = N.math("EXPONENT", N.math("MULTIPLY", r, -2.6))  # le disque s'éteint vers le bord
    thick = N.math("EXPONENT", N.math("MULTIPLY", N.math("ABSOLUTE", z), -26.0))
    arm_d = N.math("MULTIPLY", N.math("MULTIPLY", armw, clump), N.math("MULTIPLY", disk, thick))
    bulge = N.math("EXPONENT", N.math("MULTIPLY", N.math("SQRT", N.math("ADD", N.math("MULTIPLY", r, r),
                                                                           N.math("MULTIPLY", N.math("MULTIPLY", z, z), 9.0))), -9.0))
    dens = N.math("MULTIPLY", N.math("ADD", N.math("MULTIPLY", arm_d, 14.0), N.math("MULTIPLY", bulge, 6.0)), 1.0)
    # couleur : bulbe doré → bras bleus, taches roses là où le gaz est le plus dense dans les bras
    cr = N.ramp(N.math("MULTIPLY", r, 1.6), [(0.0, (1.0, 0.85, 0.55, 1)), (0.25, (1.0, 0.75, 0.5, 1)),
                                             (0.45, (0.6, 0.7, 1.0, 1)), (1.0, (0.35, 0.5, 1.0, 1))])
    hii = N.ramp(N.math("MULTIPLY", arm_d, 3.0), [(0.0, (0, 0, 0, 1)), (0.5, (0, 0, 0, 1)), (0.8, (1, 1, 1, 1))])
    col = N.new("ShaderNodeMix", data_type="RGBA")
    N.link(hii.outputs["Color"], col.inputs["Factor"])
    N.link(cr.outputs["Color"], col.inputs["A"])
    col.inputs["B"].default_value = (1.0, 0.25, 0.55, 1)
    # poussière : couloirs sombres juste à l'intérieur des bras
    dust_ph = N.math("COSINE", N.math("ADD", ph, 0.9))
    dust = N.math("MULTIPLY", N.math("POWER", N.math("MULTIPLY", N.math("ADD", dust_ph, 1.0), 0.5), 6.0),
                  N.math("MULTIPLY", disk, thick))
    vol = N.new("ShaderNodeVolumePrincipled")
    N.link(N.math("ADD", N.math("MULTIPLY", dust, 22.0), N.math("MULTIPLY", dens, 0.1)), vol.inputs["Density"])
    vol.inputs["Color"].default_value = (0.05, 0.03, 0.02, 1)
    N.link(col.outputs["Result"], vol.inputs["Emission Color"])
    N.link(N.math("MULTIPLY", dens, 2.0), vol.inputs["Emission Strength"])
    out = N.new("ShaderNodeOutputMaterial")
    N.link(vol.outputs["Volume"], out.inputs["Volume"])
    return m


def main():
    a = C.args({"--bras": dict(type=int, default=2), "--torsion": dict(type=float, default=2.2)})
    sc = C.reset()
    C.setup_render(sc, a)
    C.bloom(sc, size=8, threshold=0.6)
    C.starfield(sc, density=1.0, milky=0.05, strength=0.8)
    bpy.ops.mesh.primitive_cylinder_add(vertices=96, radius=1.0, depth=0.5)
    g = bpy.context.object
    g.scale = (6, 6, 6)
    g.data.materials.append(galaxy_material(a.bras, a.torsion, a.seed))
    g.rotation_euler = (0, 0, 0)
    g.keyframe_insert("rotation_euler", frame=1)
    g.rotation_euler = (0, 0, math.radians(-14))           # la galaxie tourne lentement
    g.keyframe_insert("rotation_euler", frame=a.frames)
    C.linear_all()
    cam = C.camera(sc, lens=30)
    C.key_path(cam, a.frames, [(0.5, -2.0, 15.0), (1.5, -9.0, 6.0), (2.0, -10.5, 1.6)],
               [(0, 0, 0), (0, 0, 0), (0.0, 0.5, 0)])
    C.render(sc, a)


main()

## Plans à rendre

In [ ]:
# ---- Plans à rendre ------------------------------------------------------------------------------
# scène : nebuleuse | planete | hyperespace | galaxie      secondes : durée du plan
# options : celles de la scène (voir l'en-tête de chaque fichier) — ex. "--palette feu", "--type glace"
# Astuce : commence par APERCU = True (petit, rapide) pour vérifier, puis passe à False pour le rendu final.
APERCU = False
PLANS = [
    ("hyperespace", 3.0, "--vitesse 1.0"),
    ("nebuleuse",   2.5, "--palette bleu --seed 1"),
    ("planete",     2.5, "--type gazeuse --inclinaison 24"),
    ("nebuleuse",   2.5, "--palette feu --seed 4"),
    ("galaxie",     3.0, "--bras 2"),
    ("planete",     2.5, "--type glace --seed 3"),
    ("nebuleuse",   2.5, "--palette rose --seed 7"),
]
ECHANTILLONS = {"nebuleuse": 128, "planete": 96, "hyperespace": 64, "galaxie": 128}

## Rendu

In [ ]:
import glob
os.makedirs(f"{ROOT}/videos", exist_ok=True)
t_all = time.time()
for i, (scene, secs, opts) in enumerate(PLANS, 1):
    frames = max(2, int(round(secs * 30)))
    name = (f"{i:02d}_{scene}_" + re.sub(r"[^a-z0-9]+", "-", opts.lower())).strip("-_")
    out_dir = f"{ROOT}/frames/{name}"
    os.makedirs(out_dir, exist_ok=True)
    samples = 24 if APERCU else ECHANTILLONS.get(scene, 96)
    res = 40 if APERCU else 100
    cmd = (f"'{BLENDER}' -b -P {ROOT}/scenes/{scene}.py -- --out {out_dir}/f_ --frames {frames} "
           f"--samples {samples} --res {res} {opts}")
    print(f"\n▶ {name} : {frames} images, {samples} échantillons")
    t0 = time.time()
    for essai, extra in enumerate(("", " --debruitage aucun")):   # plan B : sans débruitage si ça échoue
        p = subprocess.run(cmd + extra, shell=True, capture_output=True, text=True)
        log = (p.stdout + p.stderr).splitlines()
        print("\n".join(l for l in log if "[rendu]" in l or "[calibrage]" in l or "Error" in l))
        n = len(glob.glob(f"{out_dir}/f_*.png"))
        if n >= frames:
            break
        print("   ⚠ échec, dernières lignes :\n   " + "\n   ".join(log[-6:]))
        if essai == 0:
            print("   → nouvel essai sans débruitage")
    print(f"   {n} images en {time.time() - t0:.0f} s")
    if n:
        subprocess.run(f"ffmpeg -y -v error -framerate 30 -i {out_dir}/f_%04d.png -c:v libx264 -pix_fmt yuv420p "
                       f"-crf 14 -preset slow {ROOT}/videos/{name}.mp4", shell=True)
print(f"\nTerminé en {(time.time() - t_all) / 60:.1f} min →", sorted(os.listdir(f"{ROOT}/videos")))

In [ ]:
shutil.make_archive(f"{ROOT}/videos_cosmos", "zip", f"{ROOT}/videos")
print("Archive :", f"{ROOT}/videos_cosmos.zip")
try:
    from google.colab import files  # Colab : téléchargement direct
    files.download(f"{ROOT}/videos_cosmos.zip")
except ImportError:
    pass